# CADC: lung nodule malignancy on LIDC-IDRI

Run the cells from top to bottom. See the README in the repo for the full guide.

* **Step 1 (preprocess)** needs no GPU: use a CPU runtime.
* **Steps 2–4 (train, evaluate, predict)** need a GPU: *Runtime → Change runtime type → T4 GPU*.

Everything is saved to Google Drive under `MyDrive/CADC/`, and every step resumes where it stopped,
so after a disconnect or a runtime change, rerun **Setup** and continue with the step you were on.

## Setup (run this first, every session)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/CADC'
PATCHES = f'{DRIVE}/patches'   # preprocessed nodule shards
RUN = f'{DRIVE}/runs/baseline'  # checkpoints, logs and results of this training run
REPO_URL = 'https://github.com/Dhir-learner/CAD-C.git'

import os
if os.path.isdir('/content/CADC/.git'):
    !git -C /content/CADC pull -q --ff-only
else:
    !git clone -q {REPO_URL} /content/CADC
%cd /content/CADC
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo "No GPU: fine for Step 1, switch to a T4 GPU for Steps 2-4"

## Step 1 — Preprocess (once, about 2–3 hours, CPU runtime)

Downloads all 1,018 LIDC-IDRI scans from TCIA and saves a 64 mm cube around each nodule to Drive.
Scans already done are skipped, so after a disconnect just run Setup and this cell again.
**Keep the tab open and the PC awake.** The cell prints a line per scan while it runs.

In [ ]:
!python -W ignore -u -m cadc.preprocess --out {PATCHES} --work /content/dicom --prefetch 4

In [ ]:
# Check: should say 1018/1018. Failed scans are retried when you rerun the cell above.
import glob
n = len(glob.glob(f'{PATCHES}/*.npz'))
print(f'{n}/1018 scans preprocessed')
if os.path.exists(f'{PATCHES}/failed.log'):
    !grep -c "^LIDC" {PATCHES}/failed.log | xargs echo "failure entries logged:"

## Step 2 — Train (T4 GPU runtime)

Switch the runtime to a T4 GPU first, then rerun **Setup**.
Trains 5 models (one per cross-validation fold). Each fold saves a checkpoint every epoch
and resumes automatically, so if Colab disconnects or the free GPU quota runs out,
come back later, rerun Setup and this cell: finished folds are skipped.

In [ ]:
# Copy the shards to the local disk: reading them straight from Drive is slow.
!mkdir -p /content/patches && rsync -a --include='*.npz' --exclude='*' {PATCHES}/ /content/patches/
!ls /content/patches | wc -l

In [ ]:
for fold in range(5):
    !python -u -m cadc.train --data /content/patches --out {RUN} --fold {fold} --epochs 60 --batch-size 32 --workers 2

## Step 3 — Evaluate

In [ ]:
!python -m cadc.evaluate --run {RUN}
from IPython.display import Image, display
display(Image(f'{RUN}/roc.png'))

## Step 4 — Predict

Scores the nodules of one preprocessed scan with all 5 models. For a new CT scan use
`--dicom <folder> --center X Y SLICE` instead of `--shard` (see the README).

In [ ]:
!python -m cadc.predict --run {RUN} --shard {PATCHES}/LIDC-IDRI-0001_0012.npz